<a href="https://colab.research.google.com/github/solasobambo-prog/ts-data-science/blob/main/week3b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Duplicates, dtypes and string cleaning

In [1]:
import pandas as pd
import numpy as np

messy = pd.DataFrame({
    'order_id': [5001, 5002, 5003, 5003, 5004, 5005, 5006, 5006, 5007, 5008],
    'customer':   ['Ada', 'Femi', 'Zainab', 'Zainab', 'Chidi',
                   'Ada', 'Tunde', 'Tunde', 'Amaka', 'Femi'],
    'city':       ['  lagos', 'ABUJA', 'Kano ', 'Kano ', 'lagos',
                   ' Abuja', 'IBADAN', 'IBADAN', 'Lagos ', 'abuja'],
    'revenue':    ['1,250', '$800', '2,300', '2,300', '450',
                   '$1,100', '640', '640', 'N/A', '3,050'],
    'order_date': ['2026-10-05', '2026/10/06', '2026-10-07', '2026-10-07',
                   'Oct 9, 2026', '2026-10-11', 'not recorded', 'not recorded',
                   '2026-10-15', '2026-10-17'],
    'status':     ['Delivered', 'delivered', 'DELIVERED', 'DELIVERED', 'Pending',
                   'pending ', 'Cancelled', 'Cancelled', 'Delivered', 'PENDING'],
})

# Capture raw state to avoid the run-order trap
messy_raw = messy.copy()


### Notes & Predictions: Inspect & Deduplicate

* **Duplicate Prediction:** Manually checking the row values reveals that **2 rows are exact duplicates** of other rows in the dataset. Order ID 5003 (Zainab) is listed twice identically, and Order ID 5006 (Tunde) is listed twice identically. Therefore, `duplicated().sum()` should return exactly 2.
* **Deduplication Strategy Choice:** Whole-row check (`messy.drop_duplicates()`).
* **Justification:** Because every column entry matches perfectly across these redundant rows, a whole-row sweep safely eliminates the exact database insertion duplicates without accidentally stripping distinct transactions.
* **Row Count Prediction:** The initial row count is 10. Dropping the 2 unique duplicate pairs will bring our new row count to **8**.


In [2]:
# 1. Structural Diagnostics
print(f"Initial Shape: {messy.shape}")
print("\n--- DataFrame Info ---")
messy.info()

print(f"\nTotal Duplicate Rows Found: {messy.duplicated().sum()}")

# 2. Show all rows involved in duplication
print("\n--- Rows Involved in Duplication ---")
display(messy[messy.duplicated(keep=False)])

# 3. Apply Deduplication using Whole-Row Check
messy = messy.drop_duplicates().copy()

print(f"\nNew Row Count Post-Deduplication: {len(messy)}")


Initial Shape: (10, 6)

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   order_id    10 non-null     int64 
 1   customer    10 non-null     object
 2   city        10 non-null     object
 3   revenue     10 non-null     object
 4   order_date  10 non-null     object
 5   status      10 non-null     object
dtypes: int64(1), object(5)
memory usage: 612.0+ bytes

Total Duplicate Rows Found: 2

--- Rows Involved in Duplication ---


,order_id,customer,city,revenue,order_date,status
2,5003,Zainab,Kano,"2,300",2026-10-07,DELIVERED
3,5003,Zainab,Kano,"2,300",2026-10-07,DELIVERED
6,5006,Tunde,IBADAN,640,not recorded,Cancelled
7,5006,Tunde,IBADAN,640,not recorded,Cancelled



New Row Count Post-Deduplication: 8


### Notes & Predictions: Part B (Text, Types, and Dates)

*   **City & Status Count Prediction:**
    *   Currently, due to messy padding spaces and inconsistent capitalization (e.g., `'  lagos'`, `'lagos'`, `'Lagos '`), the raw data shows multiple variations for the same entity.
    *   We predict there are only **4 unique real cities** (`Lagos`, `Abuja`, `Kano`, `Ibadan`) and **3 unique real statuses** (`Delivered`, `Pending`, `Cancelled`).
*   **Order Date `NaT` Prediction:**
    *   Looking at the `order_date` column, two rows contain the text `'not recorded'`, which cannot be parsed as a date.
    *   Therefore, we predict exactly **2 `NaT` values** after running `pd.to_datetime` with `errors='coerce'`.


In [3]:
# =====================================================================
# 1. CLEANING TEXT COLUMNS (City & Status)
# =====================================================================
print(f"Initial unique cities count: {messy['city'].nunique()}")

# Clean whitespace and standardize to Title Case
messy['city'] = messy['city'].str.strip().str.title()
messy['status'] = messy['status'].str.strip().str.title()

print("\n--- Cleaned City Value Counts ---")
print(messy['city'].value_counts())

print("\n--- Cleaned Status Value Counts ---")
print(messy['status'].value_counts())


# =====================================================================
# 2. CONVERTING REVENUE TO FLOAT
# =====================================================================
# Handling the syntax break: The 'N/A' string and currency symbols ($)
# require cleaning before type casting to float.
# Note: 'N/A' values naturally become NaN when errors='coerce' is not used,
# but replacing commas and dollar signs first is necessary.
# We explicitly handle 'N/A' by converting it to np.nan.

try:
    # This attempt represents what would fail without handling 'N/A' or symbols
    # messy['revenue'].astype(float)

    # Correct handling: chain replacements to strip $, commas, and handle 'N/A'
    cleaned_revenue = (
        messy['revenue']
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .replace('N/A', np.nan)
    )
    messy['revenue'] = cleaned_revenue.astype(float)
except ValueError as e:
    print(f"Hit expected error: {e}")

print(f"\nRevenue Column Dtype: {messy['revenue'].dtype}")
print(f"Total Revenue Sum: {messy['revenue'].sum()}")


# =====================================================================
# 3. DATE TRANSFORMATIONS
# =====================================================================
# Convert dates to standard datetime format, forcing 'not recorded' strings into NaT
messy['order_date'] = pd.to_datetime(messy['order_date'], errors='coerce')

print(f"\nTotal NaT values found in order_date: {messy['order_date'].isna().sum()}")

# Derived temporal components
messy['order_month'] = messy['order_date'].dt.to_period('M')
messy['order_day'] = messy['order_date'].dt.day_name()

print("\n--- Final Cleaned DataFrame Snapshot ---")
display(messy.head(8))


Initial unique cities count: 8

--- Cleaned City Value Counts ---
city
Lagos     3
Abuja     3
Kano      1
Ibadan    1
Name: count, dtype: int64

--- Cleaned Status Value Counts ---
status
Delivered    4
Pending      3
Cancelled    1
Name: count, dtype: int64

Revenue Column Dtype: float64
Total Revenue Sum: 9590.0

Total NaT values found in order_date: 3

--- Final Cleaned DataFrame Snapshot ---


,order_id,customer,city,revenue,order_date,status,order_month,order_day
0,5001,Ada,Lagos,1250.0,2026-10-05,Delivered,2026-10,Monday
1,5002,Femi,Abuja,800.0,NaT,Delivered,NaT,NaN
2,5003,Zainab,Kano,2300.0,2026-10-07,Delivered,2026-10,Wednesday
4,5004,Chidi,Lagos,450.0,NaT,Pending,NaT,NaN
5,5005,Ada,Abuja,1100.0,2026-10-11,Pending,2026-10,Sunday
6,5006,Tunde,Ibadan,640.0,NaT,Cancelled,NaT,NaN
8,5007,Amaka,Lagos,NaN,2026-10-15,Delivered,2026-10,Thursday
9,5008,Femi,Abuja,3050.0,2026-10-17,Pending,2026-10,Saturday


### Notes & Predictions: Part C (Groupby Comparison Challenge)

* **Clean Groupby Expectation:** Grouping by our cleaned `city` column will collapse the data down into **4 distinct rows** (Lagos, Abuja, Kano, Ibadan).
* **Raw Groupby Expectation:** Grouping by the original, uncleaned city column (`messy_raw['city']`) will fail to merge identical locations due to padding spaces and casing variances (e.g., `'  lagos'`, `'lagos'`, and `'Lagos '` will each get their own separate group). We predict this will yield significantly more noise groups.
* **The Core Lesson:** Comparing the size of these two summary tables shows exactly how data pollution splits a single real-world entity into multiple fragmented buckets, breaking downstream financial reporting.


In [4]:
# =====================================================================
# 1. Clean Revenue Summary Table
# =====================================================================
# Group by cleaned city, sum revenue, reset the index to flatten the table, and sort
clean_city_revenue = (
    messy.groupby('city')['revenue']
    .sum()
    .reset_index()
    .sort_values(by='revenue', ascending=False)
)

print("--- CLEANED DATA: TOTAL REVENUE BY CITY ---")
display(clean_city_revenue)
print(f"Number of groups in cleaned report: {len(clean_city_revenue)}")


# =====================================================================
# 2. Raw Revenue Summary Table (For Comparison)
# =====================================================================
# Ensure the raw baseline column matches up with our numeric revenue column for a fair test
messy_raw['revenue_numeric'] = messy['revenue']

raw_city_revenue = (
    messy_raw.groupby('city')['revenue_numeric']
    .sum()
    .reset_index()
    .sort_values(by='revenue_numeric', ascending=False)
)

print("\n--- UNCLEANED RAW DATA: REVENUE BY CITY ---")
display(raw_city_revenue)
print(f"Number of groups in uncleaned raw report: {len(raw_city_revenue)}")


# =====================================================================
# 3. Final Comparison Insight
# =====================================================================
group_diff = len(raw_city_revenue) - len(clean_city_revenue)
print(f"\nCleaning impact: Stripping whitespace and unifying case eliminated {group_diff} duplicate noise groups!")


--- CLEANED DATA: TOTAL REVENUE BY CITY ---


,city,revenue
0,Abuja,4950.0
2,Kano,2300.0
3,Lagos,1700.0
1,Ibadan,640.0


Number of groups in cleaned report: 4

--- UNCLEANED RAW DATA: REVENUE BY CITY ---


,city,revenue_numeric
6,abuja,3050.0
4,Kano,2300.0
0,lagos,1250.0
1,Abuja,1100.0
2,ABUJA,800.0
3,IBADAN,640.0
7,lagos,450.0
5,Lagos,0.0


Number of groups in uncleaned raw report: 8

Cleaning impact: Stripping whitespace and unifying case eliminated 4 duplicate noise groups!


### Final Conclusion: The Impact of Text Standardization

Our challenge results show a clear metric improvement:
* **Uncleaned Data:** 8 fragmented groups.
* **Cleaned Data:** 4 unified groups.

#### Business Impact
Without text cleaning, data noise directly impacts business reporting. Variations like `'  lagos'`, `'lagos'`, and `'Lagos '` are treated as entirely different markets by the computer. Standardizing text columns ensures that tracking metrics roll up into the correct buckets, giving stakeholders accurate insights rather than fragmented noise.


In [5]:
messy['order_date'] = pd.to_datetime(messy['order_date'], errors='coerce', format='mixed')
print(messy['order_date'].isna().sum())   # expect 1

3
